# 00 · Setup and repo tour

**Who this is for:** you (Kelvin), getting hands-on with HeavDen-Nexus before any Azure credits are spent.

**You should already know:** basic Python and how to run a cell (`Shift+Enter`).

**By the end you will be able to:**
- confirm the notebook is using the project's own Python environment
- import project code (`generator/`) into a notebook, and reload it after editing
- find your way around the repo folders
- explain the difference between *simulated time* and *real time*, and size the data with simple arithmetic

**Outline**
1. Check the environment
2. Import project code
3. Tour the repo
4. Two clocks: simulated vs real time
5. Exercise: how many training examples?
6. Pitfalls and next steps

## 1. Check the environment

Every notebook in this repo should run on the **project's `.venv`** (created by `uv sync --all-groups`).
In VS Code, pick the kernel at the top right: *Select Kernel → Python Environments → `.venv`*.

The cell below prints which Python is running. The path should end in `.venv\Scripts\python.exe` (Windows) or `.venv/bin/python`.

In [ ]:
import sys
from pathlib import Path

print("Python version :", sys.version.split()[0])
print("Interpreter    :", sys.executable)
print("Using .venv?   :", ".venv" in sys.executable)

**What to expect:** Python `3.12.x` and `Using .venv?: True`.
If it says `False`, you're on the wrong kernel. Switch it and re-run.

## 2. Import project code

The project's Python code lives in **packages** such as `generator/`.
`uv sync` installs the project in *editable* mode, which means:

- any notebook can `import generator`, no path hacks needed
- when you edit a file in `generator/`, the notebook can pick up the change **without reinstalling**

`%autoreload` reloads edited modules automatically before each cell runs. Put it at the top of every learning notebook.

In [ ]:
%load_ext autoreload
%autoreload 2

import generator

print("generator version:", generator.__version__)
print("loaded from      :", Path(generator.__file__).resolve())

The *loaded from* path points into the repo's `generator/` folder, not a copy somewhere else.
That's the key idea of these notebooks: **you're learning with the exact code production runs.**

## 3. Tour the repo

First, find the repo root (the folder containing `pyproject.toml`), wherever this notebook is opened from.

In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for folder in [start, *start.parents]:
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("pyproject.toml not found above " + str(start))


ROOT = find_repo_root()
print("Repo root:", ROOT)

In [ ]:
FOLDERS = {
    "generator": "Synthetic hospital: patients, vitals, outcomes, drift. Runs locally.",
    "databricks": "Asset Bundle: pipelines, ML, monitoring, agent (on Azure Databricks).",
    "backend": "FastAPI API (live/demo modes), deployed to Render.",
    "web": "Next.js app (the UI users see), deployed to Vercel.",
    "docs": "Documents the chatbot searches (RAG), ADRs, case study.",
    "infra/terraform": "Azure resources around the Databricks workspace.",
    "notebooks/local": "Free learning notebooks (this one!).",
    "notebooks/databricks": "Learning notebooks that run on Databricks (cost credits).",
    "tests": "Automated tests (pytest).",
    ".github/workflows": "CI/CD and the GitHub approval workflows.",
}

for name, purpose in FOLDERS.items():
    status = "ok" if (ROOT / name).is_dir() else "MISSING"
    print(f"{name:<22} [{status:^7}]  {purpose}")

Most folders are still empty. They fill up phase by phase (see `Plan.md` §17).
The two documents to keep open while learning:
- `Plan.md`: the architecture and roadmap
- `ml_model.md`: the model explained in plain language

## 4. Two clocks: simulated vs real time

The data generator has its **own clock** (*simulated time*). It can produce days of hospital activity in a few minutes of *real time*.
Let's size one simulated day with pandas: vitals every **5 minutes**, about **300 patients**.

In [ ]:
import pandas as pd

PATIENTS = 300
READING_EVERY = "5min"

one_sim_day = pd.date_range("2026-11-15 00:00", "2026-11-15 23:55", freq=READING_EVERY)
readings_per_patient = len(one_sim_day)

print("Readings per patient per simulated day:", readings_per_patient)
print("Readings for the whole hospital per day:", f"{readings_per_patient * PATIENTS:,}")
one_sim_day[:3]

**Interpretation:** 288 readings per patient per day, about **86,400** for the hospital, which matches `ml_model.md` §4.4.
A drift episode generates 48 simulated hours in one run, so it writes about twice that, in minutes.

The model doesn't learn from every 5-minute reading. The platform rolls them up into **one example per patient per hour**.

In [ ]:
readings = pd.DataFrame({"ts": one_sim_day, "patient_id": "P-1042"})
per_hour = readings.groupby(readings["ts"].dt.floor("h")).size()

print("Hourly examples for one patient per day:", len(per_hour))
print("Readings rolled into each hourly example:", per_hour.iloc[0])

## 5. Exercise: how many training examples?

On **day 0**, the generator creates **14 simulated days** of normal history. `ml_model.md` §8 says this gives roughly 100,000 patient-hour examples.

**Your task:** compute
1. the total patient-hour examples for 14 days and 300 patients
2. how many land in each block of the time split: train = days 1–9, validation = days 10–11, test = days 12–14
3. (bonus) about how many positive examples (label = 1) there are if about 1% of patient-hours are positive (notebook 03 measures this)

Fill in the cell below, then compare with the answer cell.

In [ ]:
# Try it yourself: replace the None values
SIM_DAYS = 14
HOURS_PER_DAY = 24

total_examples = PATIENTS * SIM_DAYS * HOURS_PER_DAY  # patients x days x hours
train_examples = 9  # days 1-9
valid_examples = 2  # days 10-11
test_examples = 3  # days 12-14
positives = 0.03 * total_examples  # 3% of total

print(total_examples, train_examples, valid_examples, test_examples, positives)

In [ ]:
# Answer (run after trying)
total_examples = PATIENTS * SIM_DAYS * HOURS_PER_DAY
split_days = {"train": 9, "valid": 2, "test": 3}
per_block = {name: PATIENTS * days * HOURS_PER_DAY for name, days in split_days.items()}

print(f"total     : {total_examples:,}")
for name, n in per_block.items():
    print(f"{name:<10}: {n:,}")
print(f"positives : ~{round(total_examples * 0.01):,}  (rare events, so imbalance matters)")

## 6. Pitfalls and next steps

**Common pitfall: stale imports.** You edit `generator/...py`, re-run a cell, and nothing changes.
Fix: make sure `%autoreload 2` ran (section 2). If you renamed a function, restart the kernel.

**Second pitfall: committing outputs.** Notebook outputs can contain large data. This repo runs **nbstripout** automatically on commit, so outputs are cleared from what's committed (your local file keeps them).
On a fresh clone, run `uv run nbstripout --install --attributes .gitattributes` once.

**Optional extension:** open `Plan.md` §6.4 and work out how many readings the day-3 drift episode (48 simulated hours) writes.

**Next notebook:** `01_synthea_patients_explore`. We generate the pretend hospital's patients with Synthea and explore them.